# Import packages and the datasets

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torchsummary import summary
from pathlib import Path
from pytorch_lightning import LightningModule, Trainer
from torch.utils.data import TensorDataset, DataLoader


# Load the datasets
split_dir = Path("../data/processed/eu/splits/final")

# Load the encoded_scaled datasets
X_train_encoded_scaled = pd.read_parquet(split_dir / "X_train_encoded_scaled_final.parquet")
X_val_encoded_scaled = pd.read_parquet(split_dir / "X_val_encoded_scaled_final.parquet")
X_test_encoded_scaled = pd.read_parquet(split_dir / "X_test_encoded_scaled_final.parquet")

# Load the target datasets
y_train = pd.read_parquet(split_dir / "y_train.parquet")
y_val = pd.read_parquet(split_dir / "y_val.parquet")
y_test = pd.read_parquet(split_dir / "y_test.parquet")


# Prepare the data

In [2]:
EXTENDED_SPEC_FEATURES = [
    "m (kg)",
    "ec (cm3)",
    "ep (KW)",
    "Ft_diesel",
    "Ft_petrol",
    "Ft_petrol/electric",
    "Ft_diesel/electric",
    "Fm_H",
    "Fm_M",
    "Fm_P",
    "Electric range (km)",
    "IT_28",
    "IT_29",
    "IT_32",
    "IT_33",
    "IT_37",
    "IT_38",
    "IT_none",
]

# Select only the extended specification features from the datasets
X_train_encoded_scaled = X_train_encoded_scaled[EXTENDED_SPEC_FEATURES].values
X_val_encoded_scaled = X_val_encoded_scaled[EXTENDED_SPEC_FEATURES].values
X_test_encoded_scaled = X_test_encoded_scaled[EXTENDED_SPEC_FEATURES].values

# Convert target datasets to numpy arrays
y_train = y_train.values
y_val = y_val.values
y_test = y_test.values

# Create Tensors
X_train_tensor = torch.tensor(X_train_encoded_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_encoded_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_encoded_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Create training dataset
train_set = TensorDataset(X_train_tensor, y_train_tensor)
val_set = TensorDataset(X_val_tensor, y_val_tensor)
test_set = TensorDataset(X_test_tensor, y_test_tensor)

# Create data loaders
BATCH_SIZE = 32
train_loader = DataLoader(train_set, batch_size= BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

next(iter(train_loader))


[tensor([[-0.7407, -0.4631, -0.4352,  0.0000,  1.0000,  0.0000,  0.0000,  0.0000,
           1.0000,  0.0000, -0.4386,  1.0000,  1.0000,  0.0000,  0.0000,  0.0000,
           0.0000,  0.0000],
         [-1.3375, -0.9257, -0.7920,  0.0000,  1.0000,  0.0000,  0.0000,  0.0000,
           1.0000,  0.0000, -0.4386,  0.0000,  1.0000,  0.0000,  0.0000,  0.0000,
           0.0000,  0.0000],
         [-1.1040, -0.9257, -0.7920,  0.0000,  1.0000,  0.0000,  0.0000,  0.0000,
           1.0000,  0.0000, -0.4386,  0.0000,  1.0000,  0.0000,  0.0000,  1.0000,
           0.0000,  0.0000],
         [-0.4942, -0.7206, -0.1607,  0.0000,  1.0000,  0.0000,  0.0000,  1.0000,
           0.0000,  0.0000, -0.4386,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
           0.0000,  1.0000],
         [-0.2217,  0.2848, -0.2979,  0.0000,  1.0000,  0.0000,  0.0000,  1.0000,
           0.0000,  0.0000, -0.4386,  0.0000,  0.0000,  0.0000,  1.0000,  1.0000,
           0.0000,  0.0000],
         [-1.0131, -0.9257, -0.6959

# Prepare network

In [3]:
device ="cpu"

input_dim = X_train_tensor.shape[1]

network = nn.Sequential()
network.add_module("lin1", nn.Linear(input_dim, 100))
network.add_module("act1", nn.ReLU())
network.add_module("lin2", nn.Linear(100, 100))
network.add_module("act2", nn.ReLU())
network.add_module("lin3", nn.Linear(100, 50))
network.add_module("act3", nn.ReLU())
network.add_module("out", nn.Linear(50, 1))

network.to(device)
X_train_tensor = X_train_tensor.to(device)
y_train_tensor = y_train_tensor.to(device)
X_val_tensor = X_val_tensor.to(device)
y_val_tensor = y_val_tensor.to(device)
X_test_tensor = X_test_tensor.to(device)
y_test_tensor = y_test_tensor.to(device)

summary(network, input_size=(input_dim,), device=device);

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                  [-1, 100]           1,900
              ReLU-2                  [-1, 100]               0
            Linear-3                  [-1, 100]          10,100
              ReLU-4                  [-1, 100]               0
            Linear-5                   [-1, 50]           5,050
              ReLU-6                   [-1, 50]               0
            Linear-7                    [-1, 1]              51
Total params: 17,101
Trainable params: 17,101
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.07
Estimated Total Size (MB): 0.07
----------------------------------------------------------------


# Prepare Model

In [4]:
# RMSE Loss Function
class RMSELoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, preds, target):
        mse = torch.mean((preds - target) ** 2)
        return torch.sqrt(mse)

class MyModel(LightningModule):
    def __init__(self, network, loss_fn, optimizer):
        super().__init__()
        self.network = network
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    def forward(self, x):
        return self.network(x)

    def training_step(self, batch, batch_idx):
        X_batch, y_batch = batch
        predictions = self(X_batch)
        loss = self.loss_fn(predictions, y_batch)
        self.log(
            "train_rmse",
            loss,
            on_step=False,
            on_epoch=True,
            prog_bar=True
        )
        return loss
    
    def validation_step(self, batch, batch_idx):
        X_batch, y_batch = batch

        predictions = self(X_batch)
        y_batch = y_batch.reshape_as(predictions)

        loss = self.loss_fn(predictions, y_batch)

        self.log(
            "val_rmse",
            loss,
            on_step=False,
            on_epoch=True,
            prog_bar=True
        )

        return loss
    
    def test_step(self, batch, batch_idx):
        X_batch, y_batch = batch

        predictions = self(X_batch)
        y_batch = y_batch.reshape_as(predictions)

        rmse = self.loss_fn(predictions, y_batch)
        mae = torch.abs(predictions - y_batch).mean()

        self.log(
            "test_rmse",
            rmse,
            on_step=False,
            on_epoch=True
        )

        self.log(
            "test_mae",
            mae,
            on_step=False,
            on_epoch=True
        )

        return {
            "test_rmse": rmse,
            "test_mae": mae
        }

    def configure_optimizers(self):
        return self.optimizer

# Run the model and evaluate the results

In [5]:
model = MyModel(
    network,
    RMSELoss(),
    torch.optim.Adam(network.parameters(), lr=1e-3)
)

trainer = Trainer(
    max_epochs=100,
    log_every_n_steps=10,
    enable_progress_bar=True,
)

# train + validation
trainer.fit(
    model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader
)

# test
trainer.test(model, dataloaders=test_loader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/serhatgemici/Documents/Liora/jul26_bds_int_co2/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to e

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ network │ Sequential │ 17.1 K │ train │     0 │
│ 1 │ loss_fn │ RMSELoss   │      0 │ train │     0 │
└───┴─────────┴────────────┴────────┴───────┴───────┘

Trainable params: 17.1 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 17.1 K                                                                                               
Total estimated model params size (MB): 0.068                                                                      
Modules in train mode: 9                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/Users/serhatgemici/Documents/Liora/jul26_bds_int_co2/.venv/lib/python3.13/site-packages/pytorch_lightning/utilitie
s/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and 
treespec.is_leaf()` instead.

...

`Trainer.fit` stopped: `max_epochs=100` reached.


Output()

/Users/serhatgemici/Documents/Liora/jul26_bds_int_co2/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_mae          │    7.3694376945495605     │
│         test_rmse         │     9.909857749938965     │
└───────────────────────────┴───────────────────────────┘

[{'test_rmse': 9.909857749938965, 'test_mae': 7.3694376945495605}]